#### Importing Libraries

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from joblib import dump, load

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

processed_dir = Path("../data/processed")
models_dir = Path("../models")
artifacts_dir = Path("../artifacts")
figures_dir = Path("../reports/figures")

models_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Libraries Imported Successfully")
print(f"Random State: {RANDOM_STATE}")
print(f"Processed Data Directory: {processed_dir}")
print(f"Models Directory: {models_dir}")
print(f"Artifacts Directory: {artifacts_dir}")
print(f"Figures Directory: {figures_dir}")

Libraries Imported Successfully
Random State: 42
Processed Data Directory: ..\data\processed
Models Directory: ..\models
Artifacts Directory: ..\artifacts
Figures Directory: ..\reports\figures


#### Loading Datasets and Model Evaluation Artifacts

In [3]:
train_features_df = pd.read_csv(
    processed_dir / "train_features.csv",
    parse_dates=["arrival_datetime"]
)

test_features_df = pd.read_csv(
    processed_dir / "test_features.csv",
    parse_dates=["arrival_datetime"]
)

model_comparison_df = pd.read_csv(
    artifacts_dir / "model_comparison.csv"
)

test_comparison_df = pd.read_csv(
    artifacts_dir / "model_test_comparison.csv"
)

test_predictions_df = pd.read_csv(
    artifacts_dir / "model_test_predictions.csv",
    parse_dates=["Forecast_Date"]
)

with open(
    artifacts_dir / "selected_model_configurations.json",
    "r"
) as file:
    selected_config = json.load(file)

date_column = "arrival_datetime"
target_column = "scheduled_reservations"

feature_columns = [
    column
    for column in train_features_df.columns
    if column not in [date_column, target_column]
]

primary_model = selected_config["primary_model"]
forecast_horizon = selected_config["forecast_horizon"]

print("Datasets and Model Evaluation Artifacts Loaded Successfully")

print(f"\nTraining Features Shape: {train_features_df.shape}")
print(f"Test Features Shape: {test_features_df.shape}")
print(f"Total Predictor Features: {len(feature_columns)}")

print(f"\nPrimary Model: {primary_model}")
print(f"Forecast Horizon: {forecast_horizon} days")
print(f"Selected Models: {', '.join(selected_config['selected_models'])}")

print("\nModel Evaluation Results:")
display(test_comparison_df.round(4))

Datasets and Model Evaluation Artifacts Loaded Successfully

Training Features Shape: (405, 26)
Test Features Shape: (110, 26)
Total Predictor Features: 24

Primary Model: Weighted Averaging
Forecast Horizon: 7 days
Selected Models: Weighted Averaging, Tuned LSTM, Tuned SARIMA, Tuned Extra Trees

Model Evaluation Results:


,Model,MAE,RMSE,WAPE,R2,Forecast_Days,Evaluation_Windows
0,Tuned Extra Trees,29.0500,37.2760,32.8924,0.2056,110,16
1,Weighted Averaging,29.2473,37.6373,33.1158,0.1901,110,16
2,Tuned SARIMA,30.5649,38.6199,34.6077,0.1473,110,16
3,Tuned LSTM,32.0191,41.4961,36.2543,0.0156,110,16


**Key Observations**

- **Feature Availability:** The engineered datasets contain 405 training observations, 110 test observations, and 24 predictor features available for model explainability.
- **Model Selection:** Weighted Averaging remains the validation-selected primary model, achieving a test MAE of 29.25. Tuned Extra Trees recorded a slightly lower test MAE of 29.05, providing a useful comparison for the explainability analysis.

#### Analyzing Ensemble Component Contributions

In [5]:
ensemble_config = selected_config["weighted_averaging"]
component_weights = ensemble_config["components"]

ensemble_components_df = pd.DataFrame(
    [
        {
            "Component": model_name,
            "Weight": weight,
            "Weight_Pct": weight * 100
        }
        for model_name, weight in component_weights.items()
    ]
)

ensemble_components_df = (
    ensemble_components_df
    .sort_values("Weight", ascending=False)
    .reset_index(drop=True)
)

if not np.isclose(ensemble_components_df["Weight"].sum(), 1.0):
    raise ValueError("Ensemble component weights must sum to 1.")

ensemble_components_df.to_csv(
    artifacts_dir / "ensemble_component_weights.csv",
    index=False
)

print("Ensemble Component Analysis Completed Successfully")

print("\nEnsemble Component Weights:")
display(ensemble_components_df.round(4))

print("\nWeighted Averaging Formula:")
print(
    "Ensemble Forecast = "
    + " + ".join(
        f"({row.Weight:.2f} × {row.Component})"
        for row in ensemble_components_df.itertuples()
    )
)

print(f"\nTotal Components: {len(ensemble_components_df)}")
print(f"Total Weight: {ensemble_components_df['Weight'].sum():.2f}")

Ensemble Component Analysis Completed Successfully

Ensemble Component Weights:


,Component,Weight,Weight_Pct
0,LSTM,0.40,40.0
1,SARIMA,0.35,35.0
2,Extra Trees,0.25,25.0



Weighted Averaging Formula:
Ensemble Forecast = (0.40 × LSTM) + (0.35 × SARIMA) + (0.25 × Extra Trees)

Total Components: 3
Total Weight: 1.00


**Key Observations**

- **Ensemble Composition:** Weighted Averaging combines three forecasting approaches: LSTM (40%), SARIMA (35%), and Extra Trees (25%), integrating neural, statistical, and tree-based predictions.
- **Component Contributions:** LSTM receives the largest predefined weight, followed by SARIMA and Extra Trees. These weights determine each component's contribution to the combined forecast but do not independently establish its predictive importance or accuracy.

#### Analyzing Ensemble Component Forecast Behavior

In [6]:
component_names = list(component_weights.keys())

available_models = set(test_predictions_df["Model"].unique())

print("Checking Available Component Predictions:")

for component in component_names:
    status = (
        "Available"
        if component in available_models
        else "Not Available"
    )
    print(f"{component}: {status}")

print("\nAvailable Forecast Models:")
print(", ".join(sorted(available_models)))

print("\nEnsemble Prediction Records:")
ensemble_predictions_df = (
    test_predictions_df[
        test_predictions_df["Model"] == primary_model
    ]
    .sort_values("Forecast_Date")
    .reset_index(drop=True)
)

display(ensemble_predictions_df.head(10))

print(
    f"\nTotal Ensemble Predictions: "
    f"{len(ensemble_predictions_df)}"
)

Checking Available Component Predictions:
LSTM: Not Available
SARIMA: Not Available
Extra Trees: Not Available

Available Forecast Models:
Tuned Extra Trees, Tuned LSTM, Tuned SARIMA, Weighted Averaging

Ensemble Prediction Records:


,Model,Window,Forecast_Origin,Forecast_Date,Forecast_Step,Actual,Predicted,Absolute_Error
0,Weighted Averaging,1,2018-09-12,2018-09-13,1,109.0,85.970402,23.029598
1,Weighted Averaging,1,2018-09-12,2018-09-14,2,56.0,87.832081,31.832081
2,Weighted Averaging,1,2018-09-12,2018-09-15,3,185.0,86.628556,98.371444
3,Weighted Averaging,1,2018-09-12,2018-09-16,4,127.0,91.790600,35.209400
4,Weighted Averaging,1,2018-09-12,2018-09-17,5,58.0,87.645228,29.645228
5,Weighted Averaging,1,2018-09-12,2018-09-18,6,98.0,88.885835,9.114165
6,Weighted Averaging,1,2018-09-12,2018-09-19,7,142.0,93.960448,48.039552
7,Weighted Averaging,2,2018-09-19,2018-09-20,1,76.0,94.199195,18.199195
8,Weighted Averaging,2,2018-09-19,2018-09-21,2,109.0,92.905101,16.094899
9,Weighted Averaging,2,2018-09-19,2018-09-22,3,120.0,96.215244,23.784756



Total Ensemble Predictions: 110


**Key Observations**

- **Prediction Availability:** The evaluation artifacts contain 110 Weighted Averaging forecasts, but individual predictions from its original LSTM, SARIMA, and Extra Trees components were not saved separately.
- **Explainability Approach:** Component-level contributions must be reconstructed using the original model configurations and the same rolling-origin evaluation procedure. Tuned-model predictions cannot be treated as the ensemble's original component forecasts.

#### Preparing Original Ensemble Component Configurations

In [7]:
daily_df = pd.read_csv(
    processed_dir / "daily_reservations.csv",
    parse_dates=[date_column]
)

train_df = pd.read_csv(
    processed_dir / "train_daily.csv",
    parse_dates=[date_column]
)

test_df = pd.read_csv(
    processed_dir / "test_daily.csv",
    parse_dates=[date_column]
)

original_configs = selected_config["original_models"]

required_components = {
    "LSTM": "lstm",
    "SARIMA": "sarima",
    "Extra Trees": "extra_trees"
}

missing_configs = [
    component
    for component in required_components.values()
    if component not in original_configs
]

if missing_configs:
    raise ValueError(
        f"Missing original configurations: {missing_configs}"
    )

daily_df = daily_df.sort_values(
    date_column
).reset_index(drop=True)

train_df = train_df.sort_values(
    date_column
).reset_index(drop=True)

test_df = test_df.sort_values(
    date_column
).reset_index(drop=True)

feature_source_df = (
    pd.concat(
        [train_features_df, test_features_df],
        ignore_index=True
    )
    .sort_values(date_column)
    .reset_index(drop=True)
)

print("Original Ensemble Configurations Loaded Successfully")

print(f"\nComplete Daily Series: {daily_df.shape}")
print(f"Training Period: {train_df.shape}")
print(f"Test Period: {test_df.shape}")
print(f"Engineered Feature Dataset: {feature_source_df.shape}")

print("\nOriginal Component Configurations:")

for component_name, config_key in required_components.items():
    print(f"\n{component_name}:")
    for parameter, value in original_configs[config_key].items():
        print(f"  {parameter}: {value}")

print("\nEnsemble Component Weights:")

for component_name, weight in component_weights.items():
    print(f"{component_name}: {weight:.2%}")

Original Ensemble Configurations Loaded Successfully

Complete Daily Series: (549, 2)
Training Period: (439, 2)
Test Period: (110, 2)
Engineered Feature Dataset: (515, 26)

Original Component Configurations:

LSTM:
  lookback: 28
  input_size: 1
  hidden_size: 16
  num_layers: 1
  batch_first: True
  learning_rate: 0.001
  weight_decay: 0.001
  epochs: 100
  loss_function: MSELoss
  optimizer: Adam
  scaler: StandardScaler

SARIMA:
  order: [1, 1, 1]
  seasonal_order: [1, 0, 1, 7]
  trend: n
  enforce_stationarity: False
  enforce_invertibility: False
  maxiter: 100

Extra Trees:
  n_estimators: 200
  max_depth: 8
  min_samples_split: 10
  min_samples_leaf: 4
  max_features: 0.8
  random_state: 42
  n_jobs: -1

Ensemble Component Weights:
LSTM: 40.00%
SARIMA: 35.00%
Extra Trees: 25.00%


**Key Observations**

- **Configuration Integrity:** The original LSTM, SARIMA, and Extra Trees configurations are available, including their training parameters and the ensemble's predefined component weights.
- **Reconstruction Readiness:** The complete 549-day reservation series and 515 feature-engineered observations provide the historical inputs needed to reconstruct component predictions across the 110-day test period.

#### Defining the Original LSTM Forecasting Model

In [8]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(2)

class LSTMForecaster(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=16,
        num_layers=1,
        forecast_horizon=7
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            forecast_horizon
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])


def generate_original_lstm_forecast(history, config):
    history = np.asarray(
        history,
        dtype=np.float32
    ).reshape(-1, 1)

    lookback = config["lookback"]

    scaler = StandardScaler()
    scaled_history = scaler.fit_transform(history).ravel()

    X_sequences = []
    y_sequences = []

    for index in range(
        lookback,
        len(scaled_history) - forecast_horizon + 1
    ):
        X_sequences.append(
            scaled_history[index - lookback:index]
        )
        y_sequences.append(
            scaled_history[index:index + forecast_horizon]
        )

    X_train = torch.tensor(
        np.asarray(X_sequences, dtype=np.float32)
    ).unsqueeze(-1)

    y_train = torch.tensor(
        np.asarray(y_sequences, dtype=np.float32)
    )

    torch.manual_seed(RANDOM_STATE)

    model = LSTMForecaster(
        input_size=config["input_size"],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        forecast_horizon=forecast_horizon
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    criterion = nn.MSELoss()

    model.train()

    for epoch in range(config["epochs"]):
        optimizer.zero_grad()

        predictions = model(X_train)
        loss = criterion(predictions, y_train)

        loss.backward()
        optimizer.step()

    model.eval()

    forecast_input = torch.tensor(
        scaled_history[-lookback:],
        dtype=torch.float32
    ).reshape(1, lookback, 1)

    with torch.no_grad():
        scaled_forecast = model(
            forecast_input
        ).numpy().ravel()

    forecast = scaler.inverse_transform(
        scaled_forecast.reshape(-1, 1)
    ).ravel()

    return np.maximum(forecast, 0)


print("Original LSTM Forecasting Model Defined Successfully")
print(f"Lookback Window: {original_configs['lstm']['lookback']} days")
print(f"Forecast Horizon: {forecast_horizon} days")
print(f"Training Epochs: {original_configs['lstm']['epochs']}")

Original LSTM Forecasting Model Defined Successfully
Lookback Window: 28 days
Forecast Horizon: 7 days
Training Epochs: 100


#### Defining the Original SARIMA and Extra Trees Forecasting Models

In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def generate_original_sarima_forecast(history, config):
    history = np.asarray(
        history,
        dtype=float
    ).ravel()

    model = SARIMAX(
        history,
        order=tuple(config["order"]),
        seasonal_order=tuple(config["seasonal_order"]),
        trend=config["trend"],
        enforce_stationarity=config["enforce_stationarity"],
        enforce_invertibility=config["enforce_invertibility"]
    )

    fitted_model = model.fit(
        disp=False,
        maxiter=config["maxiter"]
    )

    forecast = fitted_model.forecast(
        steps=forecast_horizon
    )

    return np.maximum(
        np.asarray(forecast, dtype=float).ravel(),
        0
    )


def generate_original_extra_trees_forecast(
    training_features,
    forecasting_features,
    config
):
    model = ExtraTreesRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features=config["max_features"],
        random_state=config["random_state"],
        n_jobs=config["n_jobs"]
    )

    model.fit(
        training_features[feature_columns],
        training_features[target_column]
    )

    forecast = model.predict(
        forecasting_features[feature_columns]
    )

    return np.maximum(
        np.asarray(forecast, dtype=float).ravel(),
        0
    )


print("Original SARIMA and Extra Trees Models Defined Successfully")

print("\nSARIMA Configuration:")
print(f"Order: {tuple(original_configs['sarima']['order'])}")
print(
    "Seasonal Order: "
    f"{tuple(original_configs['sarima']['seasonal_order'])}"
)

print("\nExtra Trees Configuration:")
print(
    f"Estimators: "
    f"{original_configs['extra_trees']['n_estimators']}"
)
print(
    f"Maximum Depth: "
    f"{original_configs['extra_trees']['max_depth']}"
)
print(f"Predictor Features: {len(feature_columns)}")

print(f"\nForecast Horizon: {forecast_horizon} days")

Original SARIMA and Extra Trees Models Defined Successfully

SARIMA Configuration:
Order: (1, 1, 1)
Seasonal Order: (1, 0, 1, 7)

Extra Trees Configuration:
Estimators: 200
Maximum Depth: 8
Predictor Features: 24

Forecast Horizon: 7 days


#### Reconstructing Original Ensemble Component Forecasts

In [10]:
reconstructed_records = []

ensemble_reference_df = (
    test_predictions_df[
        test_predictions_df["Model"] == primary_model
    ]
    .copy()
)

ensemble_reference_df["Forecast_Date"] = pd.to_datetime(
    ensemble_reference_df["Forecast_Date"]
)

ensemble_reference_df = ensemble_reference_df.sort_values(
    ["Window", "Forecast_Date"]
).reset_index(drop=True)

window_ids = sorted(
    ensemble_reference_df["Window"].unique()
)

for window_id in window_ids:
    reference_window = (
        ensemble_reference_df[
            ensemble_reference_df["Window"] == window_id
        ]
        .sort_values("Forecast_Date")
        .reset_index(drop=True)
    )

    forecast_origin = pd.to_datetime(
        reference_window["Forecast_Origin"].iloc[0]
    )

    forecast_dates = reference_window[
        "Forecast_Date"
    ]

    history = daily_df.loc[
        daily_df[date_column] <= forecast_origin,
        target_column
    ].to_numpy()

    training_features = feature_source_df.loc[
        feature_source_df[date_column] <= forecast_origin
    ]

    forecasting_features = feature_source_df.loc[
        feature_source_df[date_column].isin(forecast_dates)
    ].sort_values(date_column)

    forecast_days = len(reference_window)

    if len(forecasting_features) != forecast_days:
        raise ValueError(
            f"Feature alignment failed for window {window_id}."
        )

    lstm_forecast = generate_original_lstm_forecast(
        history,
        original_configs["lstm"]
    )[:forecast_days]

    sarima_forecast = generate_original_sarima_forecast(
        history,
        original_configs["sarima"]
    )[:forecast_days]

    extra_trees_forecast = generate_original_extra_trees_forecast(
        training_features,
        forecasting_features,
        original_configs["extra_trees"]
    )

    reconstructed_ensemble = (
        component_weights["LSTM"] * lstm_forecast
        + component_weights["SARIMA"] * sarima_forecast
        + component_weights["Extra Trees"] * extra_trees_forecast
    )

    for index, reference_row in reference_window.iterrows():
        reconstructed_records.append({
            "Window": window_id,
            "Forecast_Date": reference_row["Forecast_Date"],
            "Forecast_Step": reference_row["Forecast_Step"],
            "Actual": reference_row["Actual"],
            "LSTM_Predicted": lstm_forecast[index],
            "SARIMA_Predicted": sarima_forecast[index],
            "Extra_Trees_Predicted": extra_trees_forecast[index],
            "Reconstructed_Ensemble": reconstructed_ensemble[index],
            "Saved_Ensemble": reference_row["Predicted"]
        })

    print(
        f"Window {window_id}/{len(window_ids)} "
        f"Reconstructed Successfully"
    )

component_forecasts_df = pd.DataFrame(
    reconstructed_records
)

component_forecasts_df["Reconstruction_Difference"] = (
    component_forecasts_df["Reconstructed_Ensemble"]
    - component_forecasts_df["Saved_Ensemble"]
)

component_forecasts_df["Absolute_Reconstruction_Difference"] = (
    component_forecasts_df["Reconstruction_Difference"].abs()
)

component_forecasts_df.to_csv(
    artifacts_dir / "ensemble_component_forecasts.csv",
    index=False
)

maximum_difference = (
    component_forecasts_df[
        "Absolute_Reconstruction_Difference"
    ].max()
)

mean_difference = (
    component_forecasts_df[
        "Absolute_Reconstruction_Difference"
    ].mean()
)

print("\nEnsemble Component Reconstruction Completed")

print(f"\nTotal Windows: {len(window_ids)}")
print(f"Total Forecasts: {len(component_forecasts_df)}")

print(f"\nMaximum Absolute Difference: {maximum_difference:.8f}")
print(f"Mean Absolute Difference: {mean_difference:.8f}")

print("\nReconstructed Forecast Sample:")
display(component_forecasts_df.head(10).round(4))

if np.allclose(
    component_forecasts_df["Reconstructed_Ensemble"],
    component_forecasts_df["Saved_Ensemble"],
    rtol=1e-5,
    atol=1e-4
):
    print("\nVerification Status: PASSED")
else:
    print("\nVerification Status: REQUIRES INVESTIGATION")

Window 1/16 Reconstructed Successfully
Window 2/16 Reconstructed Successfully
Window 3/16 Reconstructed Successfully
Window 4/16 Reconstructed Successfully
Window 5/16 Reconstructed Successfully
Window 6/16 Reconstructed Successfully
Window 7/16 Reconstructed Successfully
Window 8/16 Reconstructed Successfully
Window 9/16 Reconstructed Successfully
Window 10/16 Reconstructed Successfully
Window 11/16 Reconstructed Successfully
Window 12/16 Reconstructed Successfully
Window 13/16 Reconstructed Successfully
Window 14/16 Reconstructed Successfully
Window 15/16 Reconstructed Successfully
Window 16/16 Reconstructed Successfully

Ensemble Component Reconstruction Completed

Total Windows: 16
Total Forecasts: 110

Maximum Absolute Difference: 0.00000000
Mean Absolute Difference: 0.00000000

Reconstructed Forecast Sample:


C:\Users\user\AppData\Local\Temp\ipykernel_12976\3222445979.py:138: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(component_forecasts_df.head(10).round(4))


,Window,Forecast_Date,Forecast_Step,Actual,LSTM_Predicted,SARIMA_Predicted,Extra_Trees_Predicted,Reconstructed_Ensemble,Saved_Ensemble,Reconstruction_Difference,Absolute_Reconstruction_Difference
0,1,2018-09-13,1,109.0,90.547302,82.6793,83.2549,85.9704,85.9704,-0.0,0.0
1,1,2018-09-14,2,56.0,91.265404,86.4698,84.2459,87.8321,87.8321,0.0,0.0
2,1,2018-09-15,3,185.0,78.472801,91.8116,92.4215,86.6286,86.6286,0.0,0.0
3,1,2018-09-16,4,127.0,87.482101,93.6117,96.1346,91.7906,91.7906,0.0,0.0
4,1,2018-09-17,5,58.0,92.839401,83.7874,84.7355,87.6452,87.6452,0.0,0.0
5,1,2018-09-18,6,98.0,89.103401,88.7639,88.7083,88.8858,88.8858,0.0,0.0
6,1,2018-09-19,7,142.0,89.866600,97.5367,95.5038,93.9604,93.9604,0.0,0.0
7,2,2018-09-20,1,76.0,94.372803,96.4195,90.8131,94.1992,94.1992,0.0,0.0
8,2,2018-09-21,2,109.0,94.706299,96.1655,85.4586,92.9051,92.9051,0.0,0.0
9,2,2018-09-22,3,120.0,80.986298,112.7595,97.4196,96.2152,96.2152,0.0,0.0



Verification Status: PASSED


**Key Observations**

- **Reconstruction Accuracy:** All 110 Weighted Averaging forecasts were successfully reconstructed across 16 rolling-origin windows, with both maximum and mean absolute differences equal to zero at the displayed precision.
- **Component Verification:** The reconstructed forecasts confirm that the original LSTM (40%), SARIMA (35%), and Extra Trees (25%) predictions reproduce the saved ensemble results, establishing a reliable foundation for component-level explainability.

#### Evaluating Individual Ensemble Component Performance

In [13]:
component_prediction_columns = {
    "LSTM": "LSTM_Predicted",
    "SARIMA": "SARIMA_Predicted",
    "Extra Trees": "Extra_Trees_Predicted",
    "Weighted Averaging": "Reconstructed_Ensemble"
}

component_performance_records = []

actual_values = component_forecasts_df["Actual"].to_numpy()

for model_name, prediction_column in component_prediction_columns.items():
    predicted_values = component_forecasts_df[
        prediction_column
    ].to_numpy()

    mae = mean_absolute_error(
        actual_values,
        predicted_values
    )

    rmse = np.sqrt(
        np.mean(
            (actual_values - predicted_values) ** 2
        )
    )

    wape = (
        np.sum(np.abs(actual_values - predicted_values))
        / np.sum(np.abs(actual_values))
        * 100
    )

    component_performance_records.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "Forecast_Days": len(actual_values)
    })

component_performance_df = (
    pd.DataFrame(component_performance_records)
    .sort_values("MAE")
    .reset_index(drop=True)
)

ensemble_mae = component_performance_df.loc[
    component_performance_df["Model"] == "Weighted Averaging",
    "MAE"
].iloc[0]

component_performance_df["MAE_Improvement_vs_Ensemble"] = (
    ensemble_mae - component_performance_df["MAE"]
)

component_performance_df.to_csv(
    artifacts_dir / "ensemble_component_performance.csv",
    index=False
)

individual_components_df = component_performance_df[
    component_performance_df["Model"] != "Weighted Averaging"
]

best_individual_component = individual_components_df.iloc[0]["Model"]

print("Ensemble Component Performance Analysis Completed Successfully")

print("\nOriginal Component and Ensemble Performance:")
display(component_performance_df.round(4))

print(f"\nBest Individual Component: {best_individual_component}")
print(f"Ensemble MAE: {ensemble_mae:.4f}")
print(f"Total Models Compared: {len(component_performance_df)}")

Ensemble Component Performance Analysis Completed Successfully

Original Component and Ensemble Performance:


,Model,MAE,RMSE,WAPE,Forecast_Days,MAE_Improvement_vs_Ensemble
0,SARIMA,29.1806,37.3520,33.0403,110,0.0666
1,Weighted Averaging,29.2473,37.6373,33.1158,110,0.0000
2,Extra Trees,29.3094,37.5907,33.1862,110,-0.0622
3,LSTM,31.6098,40.9718,35.7908,110,-2.3625



Best Individual Component: SARIMA
Ensemble MAE: 29.2473
Total Models Compared: 4


**Key Observations**

- **Component Performance:** Original SARIMA achieved the lowest test MAE (29.18), marginally outperforming Weighted Averaging (29.25) and Extra Trees (29.31). Original LSTM recorded the highest MAE (31.61), despite receiving the largest ensemble weight.
- **Ensemble Effectiveness:** Weighted Averaging improved upon the individual LSTM and Extra Trees components but did not surpass SARIMA on the holdout period. The differences among SARIMA, Weighted Averaging, and Extra Trees were small, indicating broadly comparable forecasting accuracy.

#### Analyzing Ensemble Component Agreement

In [14]:
component_columns = [
    "LSTM_Predicted",
    "SARIMA_Predicted",
    "Extra_Trees_Predicted"
]

component_agreement_df = component_forecasts_df[
    ["Window", "Forecast_Date", "Forecast_Step", "Actual"]
    + component_columns
].copy()

component_agreement_df["Component_Mean"] = (
    component_agreement_df[component_columns].mean(axis=1)
)

component_agreement_df["Component_Std"] = (
    component_agreement_df[component_columns].std(
        axis=1,
        ddof=0
    )
)

component_agreement_df["Component_Range"] = (
    component_agreement_df[component_columns].max(axis=1)
    - component_agreement_df[component_columns].min(axis=1)
)

component_agreement_df["Ensemble_Predicted"] = (
    component_forecasts_df["Reconstructed_Ensemble"]
)

component_agreement_df["Ensemble_Absolute_Error"] = (
    component_agreement_df["Actual"]
    - component_agreement_df["Ensemble_Predicted"]
).abs()

agreement_summary_df = pd.DataFrame({
    "Metric": [
        "Mean Component Standard Deviation",
        "Median Component Standard Deviation",
        "Mean Component Prediction Range",
        "Maximum Component Prediction Range",
        "Correlation: Disagreement vs Absolute Error"
    ],
    "Value": [
        component_agreement_df["Component_Std"].mean(),
        component_agreement_df["Component_Std"].median(),
        component_agreement_df["Component_Range"].mean(),
        component_agreement_df["Component_Range"].max(),
        component_agreement_df[
            ["Component_Std", "Ensemble_Absolute_Error"]
        ].corr().iloc[0, 1]
    ]
})

component_agreement_df.to_csv(
    artifacts_dir / "ensemble_component_agreement.csv",
    index=False
)

agreement_summary_df.to_csv(
    artifacts_dir / "ensemble_agreement_summary.csv",
    index=False
)

print("Ensemble Component Agreement Analysis Completed Successfully")

print("\nComponent Agreement Summary:")
display(agreement_summary_df.round(4))

print("\nComponent Prediction Correlations:")
display(
    component_forecasts_df[
        component_columns
    ].corr().round(4)
)

print("\nHighest Disagreement Forecasts:")
display(
    component_agreement_df.sort_values(
        "Component_Range",
        ascending=False
    )[
        [
            "Forecast_Date",
            "Actual",
            "LSTM_Predicted",
            "SARIMA_Predicted",
            "Extra_Trees_Predicted",
            "Component_Range",
            "Ensemble_Absolute_Error"
        ]
    ].head(10).round(4)
)

Ensemble Component Agreement Analysis Completed Successfully

Component Agreement Summary:


,Metric,Value
0,Mean Component Standard Deviation,7.9687
1,Median Component Standard Deviation,6.4656
2,Mean Component Prediction Range,18.5552
3,Maximum Component Prediction Range,63.5038
4,Correlation: Disagreement vs Absolute Error,0.0819



Component Prediction Correlations:


,LSTM_Predicted,SARIMA_Predicted,Extra_Trees_Predicted
LSTM_Predicted,1.0000,0.7632,0.6201
SARIMA_Predicted,0.7632,1.0000,0.8975
Extra_Trees_Predicted,0.6201,0.8975,1.0000



Highest Disagreement Forecasts:


C:\Users\user\AppData\Local\Temp\ipykernel_12976\3592710528.py:93: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].head(10).round(4)


,Forecast_Date,Actual,LSTM_Predicted,SARIMA_Predicted,Extra_Trees_Predicted,Component_Range,Ensemble_Absolute_Error
58,2018-11-10,62.0,85.081703,120.6442,148.5855,63.5038,51.4045
37,2018-10-20,97.0,87.686798,146.9385,148.2958,60.6090,26.5772
94,2018-12-16,86.0,63.934898,87.1628,114.7572,50.8223,1.2297
80,2018-12-02,138.0,75.815804,90.1258,125.4732,49.6574,44.7613
38,2018-10-21,99.0,96.063301,137.2466,145.0904,49.0271,23.7343
59,2018-11-11,133.0,95.562500,125.0670,142.3550,46.7925,15.4128
73,2018-11-25,132.0,85.562302,98.8873,130.2809,44.7186,30.5943
41,2018-10-24,61.0,99.812897,140.9331,126.9188,41.1203,59.9815
66,2018-11-18,130.0,90.284897,107.8552,129.7170,39.4321,23.7075
51,2018-11-03,145.0,84.160103,114.1050,123.1118,38.9518,40.6212


**Key Observations**

- **Component Agreement:** SARIMA and Extra Trees exhibit the strongest prediction correlation (0.90), followed by LSTM and SARIMA (0.76). LSTM and Extra Trees show the lowest correlation (0.62), indicating comparatively greater diversity in their forecasts.
- **Forecast Disagreement:** The average prediction range across components is 18.56 reservations, reaching a maximum of 63.50. However, the correlation between component disagreement and ensemble absolute error is weak (0.08), suggesting that disagreement alone is not a reliable indicator of forecasting accuracy.